# Behavioral BAPS (no EEG)

Converted from `behavioral_baps.py`. Run cells in order. The last cell starts the interactive scorer / plots.

Data live under `experiments/LevelN/Results_CSV` and `Results_xdf` (not in git). Outputs go to `analysis/outputs/`.


In [ ]:
# Run from repo root or from analysis/. This replaces __file__ used by the .py scripts.
from pathlib import Path
import os, sys
cwd = Path.cwd().resolve()
if (cwd / "analysis").is_dir() and (cwd / "experiments").is_dir():
    REPO_ROOT = cwd
elif cwd.name == "analysis":
    REPO_ROOT = cwd.parent
elif cwd.name == "notebooks":
    REPO_ROOT = cwd.parents[1]
else:
    REPO_ROOT = cwd
_ANALYSIS_DIR = REPO_ROOT / "analysis"
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
%matplotlib inline


Behavioral Attention Analysis - BEHAVIORAL + PHYSIOLOGICAL ONLY
================================================================
CHANGES FROM ORIGINAL (eeg_analysis_all_levels.py):
  - EEG completely removed from all metric calculations and scoring
  - APS renamed to BAPS (Behavioral Attention Performance Score)
  - BAPS now uses ONLY behavioral components:
      Accuracy, RT, RT_SD, Behavioral Recovery Time
  - Expanded behavioral metrics added:
      * Inverse Efficiency Score (IES)
      * Post-Error Slowing (PES)
      * Error Clustering (Burstiness)
      * Micro-Lapse Rate (RT > 2x mean)
      * Performance Slope (within-level fatigue)
      * Recovery Shape Classification (V / U / L)
      * Recovery Completeness Rate
      * Distraction Cost (% drop from baseline)
      * Anticipatory RT Drift (pre-distraction window)
      * First Post-Distraction RT and Error
      * Cumulative Attentional Debt
      * Inter-Trial Interval Variability
  - All plots updated to be behavioral-only
  - EEG preprocessing quality audit function added (audit only, not used in scoring)
  - Physiological data (HR/HRV from Hume Band) treated as
    supplementary discussion material — loaded if available but
    NOT combined into BAPS

NOTE ON TASK DESIGN:
  Participants press SPACEBAR for correct (Go) and ANY OTHER KEY for
  No-Go. Every trial requires a keypress — there are NO omissions by
  design. Therefore:
    - response_correct = 1 means right key was pressed
    - response_correct = 0 means wrong key was pressed
    - Omissions are structurally impossible in this paradigm

EEG PREPROCESSING AUDIT (see audit_eeg_preprocessing() below):
  The original code applied:
    1. Amplitude threshold: |µV| > 500 → NaN → interpolated
    2. Bandpass filter: 1–44 Hz, 4th-order Butterworth, zero-phase
    3. (Advanced only) Notch at 60 Hz, re-reference, ICA, z-score artifact removal
  ISSUES IDENTIFIED:
    a) Threshold of ±500 µV is too lenient for Muse 2 consumer-grade data.
       Muse 2 amplitude typically stays within ±150 µV; using 500 µV
       lets through substantial muscle and motion artifacts.
    b) Bandpass goes up to 44 Hz which includes significant EMG contamination.
       For attention research (theta/alpha/beta), 1–30 Hz is more appropriate.
    c) ICA on 4 channels (TP9, AF7, AF8, TP10) is mathematically borderline —
       FastICA needs at least as many components as sources to decompose cleanly.
       With only 4 channels the solution is often unstable.
    d) Recovery baseline used 1-second pre-stim windows, which at 256 Hz is
       only 256 samples — borderline for stable spectral estimates.
    e) The ±10% epsilon threshold for EEG recovery is very loose and will
       declare "recovery" after almost any transient signal crossing.
    f) Frontal Alpha Asymmetry was computed from RMS values (not power spectra),
       which is less standard than Welch PSD-based FAA.
  CONCLUSION: EEG data quality was likely compromised by motion artifacts
  (participants actively typing during distraction conditions), making
  mid-task EEG unreliable for the primary analysis. Dropping mid-task EEG
  from the composite score is methodologically the correct decision.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import signal, stats
from scipy.stats import pearsonr, zscore, linregress
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Match the style used by survey_analysis_visualization.py (which saves correctly).
# Do NOT force a backend — the default MacOSX backend on macOS can save to file
# without opening a window as long as plt.show() is never called.
sns.set_style("whitegrid")
import matplotlib
_PALETTE = ['#4C9BE8', '#E84C4C', '#E8A94C', '#5CB85C',
            '#9B59B6', '#1ABC9C', '#E67E22', '#95A5A6']


# ══════════════════════════════════════════════════════════════════════════════
# EEG PREPROCESSING AUDIT (standalone — does not affect BAPS)
# ══════════════════════════════════════════════════════════════════════════════

def audit_eeg_preprocessing(eeg_csv_path=None, sampling_rate=256, verbose=True):
    """
    Quality audit of EEG preprocessing as originally implemented.

    Checks applied to an actual EEG CSV if provided, otherwise runs
    diagnostics on the pipeline logic alone.

    Returns a dict of findings.
    """
    findings = {}

    if verbose:
        print("\n" + "=" * 65)
        print("EEG PREPROCESSING QUALITY AUDIT")
        print("=" * 65)

    # ── Issue 1: Amplitude threshold ─────────────────────────────────
    findings['amplitude_threshold'] = {
        'original': '±500 µV',
        'recommended': '±150 µV for Muse 2',
        'severity': 'MODERATE',
        'reason': (
            'Muse 2 hardware clips at ~800 µV but clean EEG rarely '
            'exceeds ±150 µV. Using 500 µV passes substantial EMG and '
            'motion artifacts, especially during active typing.'
        )
    }

    # ── Issue 2: Bandpass upper cutoff ───────────────────────────────
    findings['bandpass_upper'] = {
        'original': '44 Hz',
        'recommended': '30 Hz for attention research',
        'severity': 'HIGH',
        'reason': (
            '30–44 Hz contains significant EMG contamination. '
            'Alpha (8–13), Beta (13–30), and Theta (4–8) bands — '
            'the ones used in TBR and ABR — are fully captured at 30 Hz.'
        )
    }

    # ── Issue 3: ICA on 4 channels ───────────────────────────────────
    findings['ica_channels'] = {
        'original': 'FastICA on 4 channels (TP9, AF7, AF8, TP10)',
        'recommended': 'Skip ICA or use only basic thresholding',
        'severity': 'HIGH',
        'reason': (
            'ICA requires independent components ≤ channels. With 4 '
            'channels the decomposition is extremely constrained and '
            'unstable. Artifact components cannot be reliably identified.'
        )
    }

    # ── Issue 4: Recovery epsilon ─────────────────────────────────────
    findings['recovery_epsilon'] = {
        'original': '±10% of baseline amplitude',
        'recommended': '±5% or fixed µV window based on noise floor',
        'severity': 'MODERATE',
        'reason': (
            '10% tolerance is very permissive. At a 30 µV baseline a '
            'signal within ±3 µV counts as recovered — this is within '
            'typical noise floor of the Muse 2.'
        )
    }

    # ── Issue 5: Motion during distraction ───────────────────────────
    findings['motion_artifacts'] = {
        'condition': 'Participants typing responses in Set 2',
        'severity': 'CRITICAL',
        'reason': (
            'Active typing produces large EMG artifacts at temporal '
            'electrodes (TP9, TP10). Motion during Set 2 distraction '
            'makes mid-task EEG in Set 2 unreliable regardless of '
            'preprocessing choices.'
        )
    }

    # ── Issue 6: FAA computation method ──────────────────────────────
    findings['faa_method'] = {
        'original': 'RMS amplitude difference after bandpass',
        'recommended': 'Welch PSD log-power difference',
        'severity': 'LOW',
        'reason': (
            'RMS is a proxy for power but not equivalent to spectral '
            'power density. Welch PSD gives more stable, standard FAA '
            'estimates comparable with literature.'
        )
    }

    # ── Run signal-level checks if data provided ─────────────────────
    if eeg_csv_path and os.path.exists(eeg_csv_path):
        try:
            df = pd.read_csv(eeg_csv_path)
            channels = [c for c in ['TP9', 'AF7', 'AF8', 'TP10'] if c in df.columns]

            if channels:
                signal_stats = {}
                for ch in channels:
                    col = df[ch].dropna()
                    pct_above_150 = (np.abs(col) > 150).mean() * 100
                    pct_above_500 = (np.abs(col) > 500).mean() * 100
                    signal_stats[ch] = {
                        'mean_abs': float(np.abs(col).mean()),
                        'max_abs': float(np.abs(col).max()),
                        'pct_above_150uV': float(pct_above_150),
                        'pct_above_500uV': float(pct_above_500),
                    }
                findings['signal_stats'] = signal_stats

                if verbose:
                    print("\nSIGNAL STATISTICS (raw data):")
                    for ch, s in signal_stats.items():
                        flag = "⚠️ HIGH ARTIFACT" if s['pct_above_150uV'] > 10 else "✓ OK"
                        print(f"  {ch}: mean|x|={s['mean_abs']:.1f}µV  "
                              f"max={s['max_abs']:.1f}µV  "
                              f">150µV: {s['pct_above_150uV']:.1f}%  {flag}")
        except Exception as e:
            findings['signal_load_error'] = str(e)

    if verbose:
        print("\nFINDINGS SUMMARY:")
        for key, val in findings.items():
            if isinstance(val, dict) and 'severity' in val:
                sev = val['severity']
                marker = {'CRITICAL': '🔴', 'HIGH': '🟠', 'MODERATE': '🟡', 'LOW': '🟢'}.get(sev, '⚪')
                print(f"\n  {marker} [{sev}] {key.upper().replace('_', ' ')}")
                for k, v in val.items():
                    if k != 'severity':
                        print(f"     {k}: {v}")

        print("\nCONCLUSION:")
        print("  Mid-task EEG from the Muse 2 headband is unreliable for")
        print("  distraction paradigms involving active typing. Dropping EEG")
        print("  from the composite score is the methodologically correct")
        print("  decision. Resting-state EEG (eyes closed, no movement) is")
        print("  more trustworthy and should be analyzed separately.")
        print("=" * 65)

    return findings


# ══════════════════════════════════════════════════════════════════════════════
# MAIN ANALYZER CLASS
# ══════════════════════════════════════════════════════════════════════════════


In [ ]:
class BehavioralAttentionAnalyzer:
    """
    Behavioral-only attention analyzer.
    EEG completely removed from scoring.
    Physiological data (Hume Band HR/HRV) loaded if available but
    kept separate — used only in supplementary discussion plots.
    """

    def __init__(self, base_path=None):
        repo = os.path.abspath(os.path.join(str(_ANALYSIS_DIR), ".."))
        if base_path is None:
            base_path = os.environ.get(
                "ATTENTIONSPAN_DATA", os.path.join(repo, "experiments")
            )
        self.base_path = base_path
        self.output_dir = os.environ.get(
            "ATTENTIONSPAN_OUTPUT", os.path.join(repo, "analysis", "outputs")
        )
        os.makedirs(self.output_dir, exist_ok=True)

        self.level1_csv_path = os.path.join(base_path, "Level1/Results_CSV")
        self.level2_csv_path = os.path.join(base_path, "Level2/Results_CSV")
        self.level3_csv_path = os.path.join(base_path, "Level3/Results_CSV")
        self.csv_path = os.path.join(base_path, "Results_CSV")

        self.level1_task_names = {1: 'Classic', 2: 'SART', 3: 'Semantic'}
        self.level2_task_names = {1: 'selectiveVisual', 2: 'colorNumber', 3: 'spatialCue'}
        self.level3_task_names = {1: 'pictureRecall', 2: 'colorLocation', 3: 'dualTask'}
        self.task_names_by_level = {
            1: self.level1_task_names,
            2: self.level2_task_names,
            3: self.level3_task_names
        }
        self.display_names = {
            'Classic': 'Classic Go/No-Go', 'SART': 'SART', 'Semantic': 'Semantic',
            'selectiveVisual': 'Selective Visual', 'colorNumber': 'Color-Number',
            'spatialCue': 'Spatial Cue', 'pictureRecall': 'Picture Recall',
            'colorLocation': 'Color-Location', 'dualTask': 'Dual Task'
        }

    # ── Path helpers ──────────────────────────────────────────────────────────

    def get_task_name(self, level_id, task_id):
        return self.task_names_by_level.get(level_id, {}).get(task_id, f'Task{task_id}')

    def get_display_name(self, task_name):
        return self.display_names.get(task_name, task_name)

    def get_csv_path_for_level(self, level_id, subject_id=None):
        paths = {1: self.level1_csv_path, 2: self.level2_csv_path, 3: self.level3_csv_path}
        base = paths.get(level_id, self.csv_path)
        if subject_id:
            sp = os.path.join(base, f"Subject{subject_id}")
            return sp if os.path.exists(sp) else sp
        return base

    def find_available_subjects(self, level_id):
        subjects = set()
        for base in [self.get_csv_path_for_level(level_id)]:
            if os.path.exists(base):
                for item in os.listdir(base):
                    if item.startswith('Subject') and os.path.isdir(os.path.join(base, item)):
                        try:
                            subjects.add(int(item.replace('Subject', '')))
                        except ValueError:
                            pass
        return sorted(subjects)

    def get_accuracy_column(self, df):
        for col in ['response_correct', 'is_correct', 'correct', 'accuracy']:
            if col in df.columns:
                return col
        return None

    # ── Data loading ──────────────────────────────────────────────────────────

    def load_behavioral_data(self, subject_id, task_name, set_id=None, level_id=1):
        csv_dir = self.get_csv_path_for_level(level_id, subject_id)
        dirs = [csv_dir]
        if level_id == 2:
            dirs.append(self.level2_csv_path)
        elif level_id == 3:
            dirs.extend([os.path.join(self.level3_csv_path, f"Subject{subject_id}"),
                         self.level3_csv_path])

        task_lower = task_name.lower()
        patterns = [
            f"{task_name}_results_set{set_id}_" if set_id else f"{task_name}_results_",
            f"{task_lower}_results_set{set_id}_" if set_id else f"{task_lower}_results_",
        ]

        for d in dirs:
            if not os.path.exists(d):
                continue
            for p in patterns:
                matching = [f for f in os.listdir(d) if p in f and f.endswith('.csv')]
                if matching:
                    # Prefer the most recent file, but duplicate re-saves are sometimes
                    # empty or a single-row debug artifact (e.g. an aborted re-run that
                    # never actually recorded the session) — skip those tiny/corrupt
                    # duplicates and fall back to the next-most-recent file with a
                    # plausible trial count rather than silently loading a near-empty
                    # CSV as if it were a real ~50-100-trial session.
                    MIN_VALID_ROWS = 10
                    for fname in sorted(matching, reverse=True):
                        fp = os.path.join(d, fname)
                        try:
                            candidate = pd.read_csv(fp)
                        except Exception as e:
                            print(f"  Skipping unreadable file {fp}: {e}")
                            continue
                        if len(candidate) < MIN_VALID_ROWS:
                            print(f"  Skipping near-empty duplicate ({len(candidate)} rows): {fp}")
                            continue
                        print(f"Loading behavioral data: {fp}")
                        return candidate

        print(f"No behavioral data: Subject {subject_id}, Level {level_id}, Task {task_name}, Set {set_id}")
        return None


    # ══════════════════════════════════════════════════════════════════════════
    # EXPANDED BEHAVIORAL METRICS
    # ══════════════════════════════════════════════════════════════════════════

    def compute_all_behavioral_metrics(self, behavioral_data, baseline_data=None):
        """
        Compute the full suite of behavioral metrics.
        Returns a dict with all metrics.

        NOTE: Because every trial requires a key press (no omissions),
        accuracy reflects wrong-key presses only — not missed trials.
        """
        if behavioral_data is None or len(behavioral_data) == 0:
            return {}

        acc_col = self.get_accuracy_column(behavioral_data)
        rt_col = 'rt' if 'rt' in behavioral_data.columns else (
            'reaction_time' if 'reaction_time' in behavioral_data.columns else None)

        if rt_col is None or acc_col is None:
            return {}

        df = behavioral_data.copy()
        rt = df[rt_col].dropna().values
        correct = df[acc_col].values

        metrics = {}

        # ── Core metrics ──────────────────────────────────────────────
        metrics['accuracy'] = float(np.nanmean(correct))
        metrics['rt_mean'] = float(np.nanmean(rt))
        metrics['rt_std'] = float(np.nanstd(rt))
        metrics['rt_cv'] = float(metrics['rt_std'] / metrics['rt_mean']) if metrics['rt_mean'] > 0 else np.nan

        # ── Inverse Efficiency Score ──────────────────────────────────
        # IES = mean RT / accuracy. Lower = more efficient.
        # Penalises fast-but-wrong and slow-but-right equally.
        acc_val = metrics['accuracy'] if metrics['accuracy'] > 0 else np.nan
        metrics['IES'] = float(metrics['rt_mean'] / acc_val) if not np.isnan(acc_val) else np.nan

        # ── Rate Correct Score ────────────────────────────────────────
        # RCS = correct responses per second of total task time
        total_time = float(np.nansum(rt))
        n_correct = int(np.nansum(correct))
        metrics['RCS'] = float(n_correct / total_time) if total_time > 0 else np.nan

        # ── Post-Error Slowing ────────────────────────────────────────
        # Average RT on trial N+1 after an error minus average RT after correct
        # Positive = brain slows down after errors (healthy self-monitoring)
        # Near-zero or negative under distraction = suppressed metacognition
        if rt_col in df.columns and acc_col in df.columns:
            post_error_rts = []
            post_correct_rts = []
            for i in range(len(df) - 1):
                next_rt = df[rt_col].iloc[i + 1]
                if np.isnan(next_rt):
                    continue
                if df[acc_col].iloc[i] == 0:
                    post_error_rts.append(next_rt)
                else:
                    post_correct_rts.append(next_rt)
            pes = (np.mean(post_error_rts) - np.mean(post_correct_rts)
                   if post_error_rts and post_correct_rts else np.nan)
            metrics['post_error_slowing'] = float(pes) if not np.isnan(pes) else np.nan
        else:
            metrics['post_error_slowing'] = np.nan

        # ── Error Clustering (Burstiness) ─────────────────────────────
        # Are errors randomly distributed or clustered?
        # Burstiness B = (σ_IEI - µ_IEI) / (σ_IEI + µ_IEI)
        # B > 0 = bursty (attentional lapses), B ≈ 0 = random, B < 0 = regular
        error_positions = np.where(correct == 0)[0]
        if len(error_positions) > 2:
            inter_error_intervals = np.diff(error_positions).astype(float)
            m, s = np.mean(inter_error_intervals), np.std(inter_error_intervals)
            metrics['error_burstiness'] = float((s - m) / (s + m)) if (s + m) > 0 else np.nan
        else:
            metrics['error_burstiness'] = np.nan

        # ── Micro-Lapse Rate ──────────────────────────────────────────
        # Proportion of trials where RT > 2x individual mean
        # Reflects brief attentional gaps even without errors
        mean_rt = metrics['rt_mean']
        micro_lapses = np.sum(rt > 2 * mean_rt)
        metrics['micro_lapse_rate'] = float(micro_lapses / len(rt)) if len(rt) > 0 else np.nan

        # ── Performance Slope (within-level fatigue) ──────────────────
        # Linear regression of RT over trial number
        # Positive slope = getting slower → fatigue / distraction accumulating
        if len(rt) >= 5:
            trial_nums = np.arange(len(df[rt_col].dropna()))
            rt_vals = df[rt_col].dropna().values
            slope, intercept, r_val, p_val, _ = linregress(trial_nums, rt_vals)
            metrics['rt_slope'] = float(slope)
            metrics['rt_slope_r2'] = float(r_val ** 2)
            metrics['rt_slope_p'] = float(p_val)
        else:
            metrics['rt_slope'] = np.nan
            metrics['rt_slope_r2'] = np.nan
            metrics['rt_slope_p'] = np.nan

        # ── ITI Variability ───────────────────────────────────────────
        # Variability of inter-trial intervals (gaps between responses)
        # High ITI variance = attentional instability even without errors
        metrics['iti_variability'] = float(np.nanstd(rt))  # RT IS the ITI here

        # ── Distraction Cost ──────────────────────────────────────────
        # % performance drop from personal baseline
        if baseline_data is not None:
            b_acc_col = self.get_accuracy_column(baseline_data)
            b_rt_col = ('rt' if 'rt' in baseline_data.columns
                        else 'reaction_time' if 'reaction_time' in baseline_data.columns
                        else None)
            if b_acc_col and b_rt_col:
                b_acc = baseline_data[b_acc_col].mean()
                b_rt = baseline_data[b_rt_col].mean()
                metrics['distraction_cost_accuracy'] = float(
                    (b_acc - metrics['accuracy']) / b_acc * 100) if b_acc > 0 else np.nan
                metrics['distraction_cost_rt'] = float(
                    (metrics['rt_mean'] - b_rt) / b_rt * 100) if b_rt > 0 else np.nan
            else:
                metrics['distraction_cost_accuracy'] = np.nan
                metrics['distraction_cost_rt'] = np.nan
        else:
            metrics['distraction_cost_accuracy'] = np.nan
            metrics['distraction_cost_rt'] = np.nan

        return metrics

    def compute_peri_distraction_metrics(self, behavioral_data, n_pre=10, n_post=15):
        """
        Compute metrics in windows BEFORE and AFTER each distraction event.

        Returns per-event data and summary statistics.

        Pre-distraction window: captures anticipatory performance drift
        Post-distraction window: captures recovery dynamics

        NOTE: No omissions possible — every trial has a response.
        """
        if behavioral_data is None:
            return {}
        if 'stim_time' not in behavioral_data.columns:
            return {}

        rt_col = ('rt' if 'rt' in behavioral_data.columns
                  else 'reaction_time' if 'reaction_time' in behavioral_data.columns
                  else None)
        acc_col = self.get_accuracy_column(behavioral_data)
        if not rt_col or not acc_col:
            return {}

        df = behavioral_data.copy()
        df['rt_smooth'] = df[rt_col].rolling(window=3, min_periods=1).mean()

        # Personal local baseline: mean RT in first 10 trials (pre-any-distraction)
        local_baseline_rt = df[rt_col].iloc[:10].mean()
        local_baseline_acc = df[acc_col].iloc[:10].mean()

        distraction_indices = df[df['stim_time'].notna()].index.tolist()
        if len(distraction_indices) == 0:
            return {}

        event_results = []

        for stim_idx in distraction_indices:
            iloc_pos = df.index.get_loc(stim_idx)

            # ── Pre-distraction window ────────────────────────────────
            pre_start = max(0, iloc_pos - n_pre)
            pre_rts = df[rt_col].iloc[pre_start:iloc_pos].dropna().values
            pre_acc = df[acc_col].iloc[pre_start:iloc_pos].values

            pre_rt_mean = np.mean(pre_rts) if len(pre_rts) > 0 else np.nan
            pre_rt_std = np.std(pre_rts) if len(pre_rts) > 1 else np.nan

            # Anticipatory drift: is RT rising before distraction?
            anticipatory_drift = np.nan
            if len(pre_rts) >= 5:
                slope, _, _, _, _ = linregress(np.arange(len(pre_rts)), pre_rts)
                anticipatory_drift = float(slope)

            # ── First post-distraction trial ──────────────────────────
            first_post_iloc = iloc_pos + 1
            first_post_rt = np.nan
            first_post_correct = np.nan
            first_post_error_type = None

            if first_post_iloc < len(df):
                first_post_rt = df[rt_col].iloc[first_post_iloc]
                first_post_correct = df[acc_col].iloc[first_post_iloc]
                if first_post_correct == 0:
                    # Classify error: impulsive (fast) vs negligent (slow)
                    if not np.isnan(first_post_rt) and not np.isnan(pre_rt_mean):
                        if first_post_rt < 0.8 * pre_rt_mean:
                            first_post_error_type = 'impulsive'
                        else:
                            first_post_error_type = 'negligent'

            # ── Recovery metrics ──────────────────────────────────────
            recovery_threshold_rt = pre_rt_mean  # recover to pre-distraction level
            if np.isnan(recovery_threshold_rt):
                recovery_threshold_rt = local_baseline_rt
            tolerance = 0.10 * recovery_threshold_rt if recovery_threshold_rt > 0 else 0.05

            post_end = min(iloc_pos + n_post + 1, len(df))
            post_rts = df['rt_smooth'].iloc[iloc_pos + 1:post_end].values
            post_acc = df[acc_col].iloc[iloc_pos + 1:post_end].values

            # Trial-to-Recovery: first trial within tolerance for 3 consecutive
            trials_to_recovery = np.nan
            recovery_shape = 'no_recovery'
            recovered = False

            consec = 0
            for j, rt_val in enumerate(post_rts):
                if np.isnan(rt_val):
                    consec = 0
                    continue
                if abs(rt_val - recovery_threshold_rt) <= tolerance:
                    consec += 1
                    if consec >= 3:
                        trials_to_recovery = float(j + 1)
                        recovered = True
                        break
                else:
                    consec = 0

            # Classify recovery shape from RT trajectory
            if len(post_rts) >= 5:
                raw_post = df[rt_col].iloc[iloc_pos + 1:post_end].values
                if recovered and trials_to_recovery <= 5:
                    recovery_shape = 'V'   # sharp drop + sharp return
                elif recovered and trials_to_recovery <= 10:
                    recovery_shape = 'U'   # gradual recovery
                elif recovered:
                    recovery_shape = 'U_slow'
                else:
                    # Check if at least trending up
                    if len(raw_post) >= 3:
                        s, _, _, _, _ = linregress(np.arange(len(raw_post)),
                                                   np.nan_to_num(raw_post, nan=recovery_threshold_rt))
                        recovery_shape = 'L' if s >= 0 else 'L_improving'
                    else:
                        recovery_shape = 'L'

            # Inter-distraction floor: mean RT in post window vs baseline
            post_floor = np.nanmean(post_rts) if len(post_rts) > 0 else np.nan
            floor_drop = float(post_floor - recovery_threshold_rt) if not np.isnan(post_floor) else np.nan

            event_results.append({
                'stim_idx': stim_idx,
                'pre_rt_mean': pre_rt_mean,
                'pre_rt_std': pre_rt_std,
                'anticipatory_drift': anticipatory_drift,
                'first_post_rt': float(first_post_rt) if not np.isnan(first_post_rt) else np.nan,
                'first_post_correct': float(first_post_correct) if not np.isnan(first_post_correct) else np.nan,
                'first_post_error_type': first_post_error_type,
                'trials_to_recovery': trials_to_recovery,
                'recovery_shape': recovery_shape,
                'recovered': recovered,
                'post_floor_rt': post_floor,
                'floor_drop': floor_drop,
            })

        if not event_results:
            return {}

        ev = event_results

        # Summary statistics
        recovery_lengths = [e['trials_to_recovery'] for e in ev if not np.isnan(e['trials_to_recovery'] or np.nan)]
        recovery_shapes = [e['recovery_shape'] for e in ev]
        shape_counts = {s: recovery_shapes.count(s) for s in set(recovery_shapes)}

        summary = {
            'n_events': len(ev),
            'mean_anticipatory_drift': float(np.nanmean([e['anticipatory_drift'] for e in ev])),
            'mean_first_post_rt': float(np.nanmean([e['first_post_rt'] for e in ev])),
            'first_post_error_rate': float(np.nanmean([1 - e['first_post_correct'] for e in ev
                                                        if not np.isnan(e['first_post_correct'] or np.nan)])),
            'mean_trials_to_recovery': float(np.mean(recovery_lengths)) if recovery_lengths else np.nan,
            'recovery_completeness_rate': float(sum(e['recovered'] for e in ev) / len(ev)),
            'shape_distribution': shape_counts,
            'mean_floor_drop': float(np.nanmean([e['floor_drop'] for e in ev])),
            'per_event': ev,
        }

        return summary

    def compute_cumulative_debt(self, behavioral_data, baseline_rt=None):
        """
        Compute cumulative attentional debt across the session.

        For every trial where RT exceeds the personal baseline,
        the excess is added to a running debt counter.
        Debt resets when performance returns to baseline.

        Returns trial-by-trial debt series and peak debt.
        """
        rt_col = ('rt' if 'rt' in behavioral_data.columns
                  else 'reaction_time' if 'reaction_time' in behavioral_data.columns
                  else None)
        if not rt_col:
            return np.array([]), np.nan

        rt_vals = behavioral_data[rt_col].values
        if baseline_rt is None:
            baseline_rt = np.nanmean(rt_vals[:max(10, len(rt_vals) // 5)])

        debt = []
        running_debt = 0.0
        for rt_val in rt_vals:
            if np.isnan(rt_val):
                debt.append(running_debt)
                continue
            excess = rt_val - baseline_rt
            if excess > 0:
                running_debt += excess
            else:
                # Partial debt repayment — reduce by the amount RT was below baseline
                running_debt = max(0.0, running_debt + excess)
            debt.append(running_debt)

        debt_array = np.array(debt)
        return debt_array, float(np.max(debt_array)) if len(debt_array) > 0 else np.nan

    # ── Behavioral Recovery Time (same as original, kept for BAPS) ────────────

    def compute_behavioral_recovery(self, behavioral_data):
        """Compute mean behavioral recovery time (sum of real RTs during recovery)."""
        rt_col = ('rt' if 'rt' in behavioral_data.columns
                  else 'reaction_time' if 'reaction_time' in behavioral_data.columns
                  else None)
        if not rt_col or 'stim_time' not in behavioral_data.columns:
            return np.nan

        distraction_indices = behavioral_data[behavioral_data['stim_time'].notna()].index.tolist()
        if len(distraction_indices) == 0:
            return np.nan

        df = behavioral_data.copy()
        df['rt_smooth'] = df[rt_col].rolling(window=3, min_periods=1).mean()
        recovery_times = []

        for dist_idx in distraction_indices:
            iloc_pos = df.index.get_loc(dist_idx)
            if iloc_pos >= len(df) - 3:
                continue
            baseline_start = max(0, iloc_pos - 3)
            baseline_rt = df['rt_smooth'].iloc[baseline_start:iloc_pos].mean()
            if np.isnan(baseline_rt) or baseline_rt <= 0:
                continue

            elapsed = 0.0
            for j in range(iloc_pos + 1, len(df)):
                rt_j = df['rt_smooth'].iloc[j]
                if np.isnan(rt_j):
                    continue
                elapsed += df[rt_col].iloc[j]
                if abs(rt_j - baseline_rt) < 0.10 * baseline_rt:
                    recovery_times.append(elapsed)
                    break

        return float(np.mean(recovery_times)) if recovery_times else np.nan

    def compute_trials_to_consecutive_correct(self, behavioral_data, num_consecutive=3):
        """Trials to N consecutive correct responses after each distraction."""
        acc_col = self.get_accuracy_column(behavioral_data)
        if acc_col is None or 'stim_time' not in behavioral_data.columns:
            return {'mean': np.nan, 'per_stimulus': []}

        stim_indices = behavioral_data[behavioral_data['stim_time'].notna()].index.tolist()
        trials_to_recovery = []

        for stim_idx in stim_indices:
            iloc_pos = behavioral_data.index.get_loc(stim_idx)
            consec = 0
            trials_needed = 0
            for i in range(iloc_pos + 1, len(behavioral_data)):
                trials_needed += 1
                if behavioral_data[acc_col].iloc[i] == 1:
                    consec += 1
                    if consec >= num_consecutive:
                        trials_to_recovery.append(trials_needed)
                        break
                else:
                    consec = 0
            else:
                trials_to_recovery.append(np.nan)

        valid = [t for t in trials_to_recovery if not np.isnan(t)]
        return {'mean': float(np.mean(valid)) if valid else np.nan,
                'per_stimulus': trials_to_recovery}

    # ══════════════════════════════════════════════════════════════════════════
    # BAPS — BEHAVIORAL ATTENTION PERFORMANCE SCORE
    # ══════════════════════════════════════════════════════════════════════════

    def compute_baps(self, behavioral_data, recovery_time=None,
                     trials_to_3cc=None,
                     baseline_metrics=None, is_baseline=False,
                     extended_metrics=None):
        """
        Returns (BAPS_s, BAPS_3CC, BAPS_Extended, raw_metrics) on baseline,
        or  (BAPS_s, BAPS_3CC, BAPS_Extended) on distraction sets.

        BAPS_s        — Accuracy + RT + RT_SD + Recovery_s (4 components).
                        Recovery_s = elapsed seconds until RT returns to within
                        10 % of pre-distraction RT; scale 0 s → 1.0, ≥5 s → 0.0.

        BAPS_3CC      — Accuracy + RT + RT_SD + Recovery_3CC (4 components).
                        Recovery_3CC = trials needed for 3 consecutive correct
                        responses after each distraction.
                        Normalization: score = 3 / max(mean_trials, 3).
                        • Achieved immediately (3 trials) → score = 1.0
                        • 6 trials → 0.50, 9 trials → 0.33, 15 trials → 0.20
                        The formula is a harmonic decay with no hard ceiling —
                        more trials always lowers the score proportionally.

        BAPS_Extended — Accuracy + RT + RT_SD + Recovery_s + IES +
                        Post-Error Slowing + Micro-Lapse Rate + Burstiness.
                        Richer cognitive-efficiency view; not normalized against
                        baseline for error-pattern components (absolute scale).

        All score components normalized against Set 1 (Baseline) values stored
        from the first pass through the analysis loop except where noted.
        Baseline always returns 100 for all three scores.
        Scores > 100 are allowed when the participant outperforms baseline.
        """
        rt_col = ('rt' if behavioral_data is not None and 'rt' in behavioral_data.columns
                  else 'reaction_time' if behavioral_data is not None
                       and 'reaction_time' in behavioral_data.columns
                  else None)
        acc_col = self.get_accuracy_column(behavioral_data) if behavioral_data is not None else None

        if rt_col is None or acc_col is None or behavioral_data is None:
            return (100.0, 100.0, 100.0, {}) if is_baseline else (0.0, 0.0, 0.0)

        Accuracy = float(behavioral_data[acc_col].mean())
        RT       = float(behavioral_data[rt_col].mean())
        RT_std   = float(behavioral_data[rt_col].std())
        raw_metrics = {'Accuracy': Accuracy, 'RT': RT, 'RT_std': RT_std}

        if is_baseline:
            print(f"   BASELINE — BAPS_s = BAPS_3CC = BAPS_Extended = 100 (reference)")
            print(f"   Cached: Acc={Accuracy:.1%}  RT={RT:.3f}s  RT_SD={RT_std:.3f}s")
            return 100.0, 100.0, 100.0, raw_metrics

        if baseline_metrics is None:
            print("   Warning: no baseline metrics — BAPS = 0")
            return 0.0, 0.0, 0.0

        em = extended_metrics or {}

        # ── Shared sub-scores: Accuracy, RT, RT_SD ───────────────────────────
        shared_comps, shared_labels = [], []

        b_acc = baseline_metrics.get('Accuracy', np.nan)
        if not np.isnan(Accuracy) and not np.isnan(b_acc) and b_acc > 0:
            s = Accuracy / b_acc
            shared_comps.append(s); shared_labels.append('Accuracy')
            print(f"   [1] Accuracy:     {s:.3f}  ({Accuracy:.1%} vs baseline {b_acc:.1%})")

        b_rt = baseline_metrics.get('RT', np.nan)
        if not np.isnan(RT) and not np.isnan(b_rt) and b_rt > 0 and RT > 0:
            s = b_rt / RT
            shared_comps.append(s); shared_labels.append('RT')
            print(f"   [2] RT:           {s:.3f}  ({RT:.3f}s vs baseline {b_rt:.3f}s)")

        b_rt_std = baseline_metrics.get('RT_std', np.nan)
        if not np.isnan(RT_std) and not np.isnan(b_rt_std) and b_rt_std > 0 and RT_std > 0:
            s = b_rt_std / RT_std
            shared_comps.append(s); shared_labels.append('RT_SD')
            print(f"   [3] RT_SD:        {s:.3f}  ({RT_std:.3f}s vs baseline {b_rt_std:.3f}s)")

        # ── BAPS_s: shared + Recovery_seconds ────────────────────────────────
        # Recovery_s = sum of actual RTs from distraction trial until rolling
        # RT mean returns to within 10 % of the 3-trial pre-distraction mean.
        # Baseline-relative scale: ceiling = RECOVERY_RT_MULTIPLIER x this
        # participant's OWN baseline RT (b_rt, already computed above for the
        # RT component) rather than one universal constant for every participant —
        # consistent with Accuracy/RT/RT_SD all being scored against that same
        # participant's Set 1 values instead of a fixed threshold.
        RECOVERY_RT_MULTIPLIER = 10.0
        if recovery_time is None or (isinstance(recovery_time, float) and np.isnan(recovery_time)):
            rec_s_score = 0.0
            print(f"   [4s] Recovery_s:  0.000  (no RT recovery detected — penalized)")
        elif not np.isnan(b_rt) and b_rt > 0:
            rec_val = float(recovery_time)
            recovery_ceiling_s = RECOVERY_RT_MULTIPLIER * b_rt
            rec_s_score = max(0.0, 1.0 - rec_val / recovery_ceiling_s)
            print(f"   [4s] Recovery_s:  {rec_s_score:.3f}  "
                  f"({rec_val:.3f}s vs baseline-relative ceiling {recovery_ceiling_s:.2f}s = "
                  f"{RECOVERY_RT_MULTIPLIER:.0f}x baseline RT {b_rt:.3f}s)")
        else:
            rec_s_score = 0.0
            print(f"   [4s] Recovery_s:  0.000  (baseline RT unavailable — cannot "
                  f"compute an individualized ceiling, penalized)")

        s_comps  = shared_comps + [rec_s_score]
        s_labels = shared_labels + ['Recovery_s']
        baps_s = float(np.mean(s_comps)) * 100 if s_comps else 0.0
        print(f"\n   BAPS_s = mean({len(s_comps)}) × 100 = {baps_s:.2f}  "
              f"(components: {s_labels})")

        # ── BAPS_3CC: shared + Recovery_3CC ──────────────────────────────────
        # Recovery_3CC = mean trials to 3 consecutive correct responses after
        # each distraction event, averaged across all distractions in the set.
        #
        # Normalization: score = 3 / max(mean_trials, 3)
        #   • The minimum possible is 3 (achieved the 3 correct immediately).
        #     → score = 3/3 = 1.0  (perfect resilience)
        #   • 6 trials  → 3/6  = 0.50
        #   • 9 trials  → 3/9  = 0.33
        #   • 15 trials → 3/15 = 0.20
        #   • NaN (never recovered within the session) → score = 0.0 (penalty)
        #
        # No hard ceiling needed: the harmonic decay naturally approaches 0
        # as trials increase, so very slow recovery is heavily penalized.
        t3cc = trials_to_3cc  # mean across distraction events in this set
        if t3cc is None or (isinstance(t3cc, float) and np.isnan(t3cc)):
            rec_3cc_score = 0.0
            print(f"   [4c] Recovery_3CC: 0.000  (3CC never achieved — penalized)")
        else:
            t3cc_val = float(t3cc)
            rec_3cc_score = 3.0 / max(t3cc_val, 3.0)
            print(f"   [4c] Recovery_3CC: {rec_3cc_score:.3f}  "
                  f"({t3cc_val:.1f} trials avg to 3CC,  score = 3/{max(t3cc_val,3):.1f})")

        cc_comps  = shared_comps + [rec_3cc_score]
        cc_labels = shared_labels + ['Recovery_3CC']
        baps_3cc = float(np.mean(cc_comps)) * 100 if cc_comps else 0.0
        print(f"   BAPS_3CC = mean({len(cc_comps)}) × 100 = {baps_3cc:.2f}  "
              f"(components: {cc_labels})")

        # ── BAPS_Extended: BAPS_s base + IES + PostError + Lapses + Burst ────
        ext_comps  = list(s_comps)
        ext_labels = list(s_labels)

        IES   = em.get('IES', np.nan)
        b_IES = baseline_metrics.get('IES', np.nan)
        if not np.isnan(IES) and not np.isnan(b_IES) and b_IES > 0 and IES > 0:
            s = b_IES / IES
            ext_comps.append(s); ext_labels.append('IES')
            print(f"   [5]  IES:          {s:.3f}  ({IES:.3f} vs baseline {b_IES:.3f})")

        PES = em.get('post_error_slowing', np.nan)
        if not np.isnan(PES):
            s = 1.0 / (1.0 + abs(PES))
            ext_comps.append(s); ext_labels.append('PostError')
            print(f"   [6]  PostError:    {s:.3f}  (PES={PES:.3f}s)")

        lapse = em.get('micro_lapse_rate', np.nan)
        if not np.isnan(lapse):
            s = max(0.0, 1.0 - float(lapse))
            ext_comps.append(s); ext_labels.append('Lapses')
            print(f"   [7]  Lapses:       {s:.3f}  (rate={lapse:.1%})")

        burst = em.get('error_burstiness', np.nan)
        if not np.isnan(burst):
            s = max(0.0, 1.0 - abs(float(burst)))
            ext_comps.append(s); ext_labels.append('Burstiness')
            print(f"   [8]  Burstiness:   {s:.3f}  (B={burst:.3f})")

        baps_ext = float(np.mean(ext_comps)) * 100 if ext_comps else 0.0
        print(f"   BAPS_Extended = mean({len(ext_comps)}) × 100 = {baps_ext:.2f}  "
              f"(components: {ext_labels})")

        return baps_s, baps_3cc, baps_ext

    # ══════════════════════════════════════════════════════════════════════════
    # PLOTTING
    # ══════════════════════════════════════════════════════════════════════════

    def plot_baps_by_condition(self, results_df, save_dir=None):
        """
        Plot 1: BAPS bar chart grouped by task and condition.
        Three bars per task (Baseline / Active / Passive), three task groups per level.
        """
        if results_df is None or len(results_df) == 0:
            return

        conditions = ['Baseline', 'Distraction+Response', 'Distraction+NoResponse']
        colors = {'Baseline': '#4C9BE8', 'Distraction+Response': '#E84C4C',
                  'Distraction+NoResponse': '#E8A94C'}

        fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
        fig.suptitle('Behavioral Attention Performance Score (BAPS) by Task and Distraction Type',
                     fontsize=14, fontweight='bold', y=1.02)

        for ax_idx, level_id in enumerate([1, 2, 3]):
            ax = axes[ax_idx]
            level_data = results_df[results_df['Level'] == level_id]
            tasks = level_data['Task'].unique()
            x = np.arange(len(tasks))
            width = 0.25

            for c_idx, condition in enumerate(conditions):
                cond_data = level_data[level_data['Condition'] == condition]
                vals = [cond_data[cond_data['Task'] == t]['BAPS_s'].values[0]
                        if len(cond_data[cond_data['Task'] == t]) > 0 else np.nan
                        for t in tasks]
                bars = ax.bar(x + c_idx * width, vals, width,
                              label=condition, color=colors[condition], alpha=0.85,
                              edgecolor='white', linewidth=0.8)
                for bar, val in zip(bars, vals):
                    if not np.isnan(val):
                        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                                f'{val:.0f}', ha='center', va='bottom', fontsize=7)

            ax.axhline(100, color='black', linestyle='--', linewidth=0.8, alpha=0.5,
                       label='Baseline reference')
            ax.set_title(f'Level {level_id}', fontweight='bold')
            ax.set_xlabel('Task')
            ax.set_ylabel('BAPS (%)') if ax_idx == 0 else None
            ax.set_xticks(x + width)
            ax.set_xticklabels(tasks, rotation=20, ha='right', fontsize=8)
            ax.set_ylim(0, 160)
            ax.grid(axis='y', alpha=0.3, linestyle='--')
            ax.legend(fontsize=7, loc='upper right')

        plt.tight_layout()
        return self._save_or_show(fig, save_dir, 'plot1_baps_by_condition.png')

    def plot_individual_heatmap(self, results_df, save_dir=None):
        """
        Plot 2: Individual BAPS heatmap.
        Rows = participants, Columns = task × condition combinations.
        Scores above 100 shown in green, below 100 in red gradient.
        """
        if results_df is None or len(results_df) == 0:
            return

        pivot = results_df.pivot_table(
            values='BAPS_s',
            index='Participant',
            columns=['Level', 'Condition', 'Task'],
            aggfunc='mean'
        )

        fig, ax = plt.subplots(figsize=(max(14, pivot.shape[1] * 0.9), max(6, pivot.shape[0] * 0.6)))

        # Custom diverging colormap centered at 100
        from matplotlib.colors import TwoSlopeNorm
        vmin, vcenter, vmax = 30, 100, 160
        norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
        cmap = plt.get_cmap('RdYlGn')

        im = ax.imshow(pivot.values, aspect='auto', cmap=cmap, norm=norm)
        plt.colorbar(im, ax=ax, label='BAPS (%)', shrink=0.8)

        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels([f'P{p}' for p in pivot.index], fontsize=9)

        col_labels = [f'L{l}\n{c[:3]}\n{t[:4]}' for l, c, t in pivot.columns]
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(col_labels, fontsize=7, rotation=45, ha='right')

        # Annotate cells
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                            fontsize=6, color='black' if 60 < val < 130 else 'white',
                            fontweight='bold' if val > 100 else 'normal')

        ax.set_title('Individual BAPS Heatmap\n(Green > 100% = above baseline; Red = below baseline)',
                     fontweight='bold', fontsize=12)
        ax.set_xlabel('Level / Condition / Task')
        ax.set_ylabel('Participant')
        plt.tight_layout()
        return self._save_or_show(fig, save_dir, 'plot2_individual_heatmap.png')

    def plot_recovery_time(self, results_df, save_dir=None):
        """
        Plot 3: Recovery time bar chart by task and distraction type.
        Shows mean behavioral recovery time (seconds) for Sets 2 and 3.
        """
        if results_df is None or len(results_df) == 0:
            return

        dist_data = results_df[results_df['Condition'] != 'Baseline'].copy()
        if len(dist_data) == 0:
            return

        fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
        fig.suptitle('Behavioral Recovery Time by Task and Distraction Type',
                     fontsize=15, fontweight='bold')

        colors = {'Distraction+Response': '#E84C4C', 'Distraction+NoResponse': '#E8A94C'}

        for ax_idx, level_id in enumerate([1, 2, 3]):
            ax = axes[ax_idx]
            level_data = dist_data[dist_data['Level'] == level_id]
            tasks = level_data['Task'].unique()
            x = np.arange(len(tasks))
            width = 0.35

            for c_idx, condition in enumerate(['Distraction+Response', 'Distraction+NoResponse']):
                cond_data = level_data[level_data['Condition'] == condition]
                vals = [cond_data[cond_data['Task'] == t]['Recovery_s'].values[0]
                        if len(cond_data[cond_data['Task'] == t]) > 0 else np.nan
                        for t in tasks]
                bars = ax.bar(x + c_idx * width, vals, width,
                              label=condition, color=colors[condition], alpha=0.85,
                              edgecolor='white')
                for bar, val in zip(bars, vals):
                    if not np.isnan(val) and val > 0:
                        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                                f'{val:.1f}s', ha='center', va='bottom',
                                fontsize=9, fontweight='bold')

            ax.set_title(f'Level {level_id}', fontweight='bold', fontsize=13)
            ax.set_xlabel('Task', fontsize=11, fontweight='bold')
            ax.set_ylabel('Recovery Time (s)', fontsize=11, fontweight='bold') if ax_idx == 0 else None
            ax.set_xticks(x + width / 2)
            ax.set_xticklabels(tasks, rotation=25, ha='right', fontsize=10, fontweight='bold')
            ax.tick_params(axis='y', labelsize=10)
            ax.grid(axis='y', alpha=0.3, linestyle='--')
            ax.legend(fontsize=9, framealpha=0.8)

        plt.tight_layout()
        return self._save_or_show(fig, save_dir, 'plot3_recovery_time.png')

    def plot_performance_trajectory(self, behavioral_data_by_session, save_dir=None):
        """
        Plot 4: RT at distraction trials only (scatter, no connecting lines).
        One figure per level. Trial index is the position in the concatenated
        session (tasks in analysis order). Baseline (set 1) usually has no
        distraction-marked trials. Files: plot4_performance_trajectory_L1.png …
        """
        if not behavioral_data_by_session:
            return False

        colors = {'Baseline': '#4C9BE8', 'Distraction+Response': '#E84C4C',
                  'Distraction+NoResponse': '#E8A94C'}
        condition_map = {1: 'Baseline', 2: 'Distraction+Response', 3: 'Distraction+NoResponse'}

        def _stim_col(df):
            return next((c for c in df.columns if str(c).lower() == 'stim_time'), None)

        def _collect_distraction_scatter(dfs, set_id):
            xs, ys = [], []
            seq = 0
            for df in dfs:
                if df is None or len(df) == 0:
                    continue
                rt_col = 'rt' if 'rt' in df.columns else (
                    'reaction_time' if 'reaction_time' in df.columns else None)
                if not rt_col:
                    continue
                stim_c = _stim_col(df)
                dfr = df.reset_index(drop=True)
                for i in range(len(dfr)):
                    row = dfr.iloc[i]
                    is_dist = False
                    if stim_c is not None and set_id in (2, 3):
                        v = row[stim_c]
                        if pd.notna(v) and str(v).strip().lower() not in ('', 'nan', 'none'):
                            is_dist = True
                    if is_dist:
                        xs.append(seq + i)
                        ys.append(float(row[rt_col]))
                seq += len(dfr)
            return xs, ys

        global_ymin, global_ymax = np.inf, -np.inf
        level_cache = {}
        for level_id in [1, 2, 3]:
            level_cache[level_id] = {}
            for set_id in [1, 2, 3]:
                key = (level_id, set_id)
                if key not in behavioral_data_by_session:
                    continue
                xs, ys = _collect_distraction_scatter(behavioral_data_by_session[key], set_id)
                if ys:
                    level_cache[level_id][set_id] = (xs, ys)
                    global_ymin = min(global_ymin, np.nanmin(ys))
                    global_ymax = max(global_ymax, np.nanmax(ys))

        if not any(level_cache[l] for l in [1, 2, 3]) or not np.isfinite(global_ymin):
            return False

        any_saved = False
        for level_id in [1, 2, 3]:
            if not level_cache[level_id]:
                continue
            fig, ax = plt.subplots(figsize=(6, 5))
            for set_id in [1, 2, 3]:
                if set_id not in level_cache[level_id]:
                    continue
                xs, ys = level_cache[level_id][set_id]
                if not ys:
                    continue
                condition = condition_map[set_id]
                ax.scatter(
                    xs,
                    ys,
                    c=colors[condition],
                    s=28,
                    alpha=0.78,
                    edgecolors='white',
                    linewidths=0.35,
                    label=condition,
                )
            ax.set_title(f'Level {level_id}', fontweight='bold')
            ax.set_xlabel('Trial (concatenated session)')
            ax.set_ylabel('RT at distraction trial (s)')
            ax.set_ylim(global_ymin * 0.95, global_ymax * 1.05)
            ax.grid(alpha=0.3, linestyle='--')
            ax.legend(fontsize=7)
            plt.tight_layout()
            saved = self._save_or_show(fig, save_dir,
                                       f'plot4_performance_trajectory_L{level_id}.png')
            any_saved = any_saved or saved

        return any_saved

    def plot_subjective_vs_objective(self, results_df, survey_data, save_dir=None):
        """
        Plot 5: Scatter of self-reported distraction vs objective BAPS.
        Annotates outlier participants.
        Includes regression line and r-value.
        """
        if survey_data is None or results_df is None:
            print("No survey data for subjective vs objective plot.")
            return

        # Average BAPS across distraction sets per participant
        dist_baps = (results_df[results_df['Condition'] != 'Baseline']
                     .groupby('Participant')['BAPS_s'].mean().reset_index())
        dist_baps.columns = ['Participant', 'avg_baps']

        merged = dist_baps.merge(survey_data, on='Participant', how='inner')
        if len(merged) < 3:
            print("Insufficient merged data for subjective vs objective plot.")
            return

        survey_col = [c for c in merged.columns if 'distract' in c.lower() or 'felt' in c.lower()]
        if not survey_col:
            survey_col = [c for c in merged.columns if c != 'Participant' and c != 'avg_baps']
        if not survey_col:
            return

        distraction_col = survey_col[0]
        x = merged[distraction_col].values
        y = merged['avg_baps'].values

        fig, ax = plt.subplots(figsize=(8, 6))

        # Scatter colored by distraction level
        scatter = ax.scatter(x, y, c=x, cmap='RdYlGn_r', s=100, zorder=3,
                             edgecolors='gray', linewidths=0.8)
        plt.colorbar(scatter, ax=ax, label='Self-reported distraction level')

        # Regression line
        if len(x) >= 3:
            slope, intercept, r_val, p_val, _ = linregress(x, y)
            x_line = np.linspace(x.min(), x.max(), 50)
            ax.plot(x_line, slope * x_line + intercept, 'k--', linewidth=1.5,
                    label=f'r = {r_val:.3f}, p = {p_val:.3f}')

        # Annotate outliers (> 1.5 SD from regression)
        residuals = y - (slope * x + intercept)
        outlier_threshold = 1.5 * np.std(residuals)
        for _, row in merged.iterrows():
            res = row['avg_baps'] - (slope * row[distraction_col] + intercept)
            if abs(res) > outlier_threshold:
                ax.annotate(f"P{row['Participant']}",
                            xy=(row[distraction_col], row['avg_baps']),
                            xytext=(5, 5), textcoords='offset points',
                            fontsize=9, color='darkred', fontweight='bold')

        ax.set_xlabel('Self-Reported Distraction Level (Likert)', fontsize=11)
        ax.set_ylabel('Mean BAPS (%) — Distraction Conditions', fontsize=11)
        ax.set_title('Subjective Distraction vs Objective Performance\n'
                     'Outliers labeled — misalignment reveals metacognitive gaps',
                     fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3, linestyle='--')
        plt.tight_layout()
        return self._save_or_show(fig, save_dir, 'plot5_subjective_vs_objective.png')

    def plot_recovery_shape_distribution(self, peri_distraction_results, save_dir=None):
        """
        Plot 6: Recovery shape distribution (V / U / L) across conditions.
        """
        if not peri_distraction_results:
            return

        shape_data = {}
        for (level_id, set_id, task_name), peri in peri_distraction_results.items():
            if set_id == 1:
                continue
            condition = 'Active' if set_id == 2 else 'Passive'
            if condition not in shape_data:
                shape_data[condition] = {'V': 0, 'U': 0, 'U_slow': 0, 'L': 0, 'L_improving': 0, 'no_recovery': 0}
            dist = peri.get('shape_distribution', {})
            for shape, count in dist.items():
                if shape in shape_data[condition]:
                    shape_data[condition][shape] += count

        if not shape_data:
            return

        fig, axes = plt.subplots(1, len(shape_data), figsize=(5 * len(shape_data), 5))
        if len(shape_data) == 1:
            axes = [axes]

        shape_colors = {'V': '#2ECC71', 'U': '#F39C12', 'U_slow': '#E67E22',
                        'L': '#E74C3C', 'L_improving': '#C0392B', 'no_recovery': '#7F8C8D'}
        shape_labels = {'V': 'V-shape\n(rapid rebound)', 'U': 'U-shape\n(gradual)',
                        'U_slow': 'U-slow\n(effortful)', 'L': 'L-shape\n(no recovery)',
                        'L_improving': 'L-improve\n(trending)', 'no_recovery': 'No recovery'}

        for ax, (condition, counts) in zip(axes, shape_data.items()):
            filtered = {k: v for k, v in counts.items() if v > 0}
            if not filtered:
                continue
            wedge_colors = [shape_colors.get(k, '#BDC3C7') for k in filtered.keys()]
            wedge_labels = [shape_labels.get(k, k) for k in filtered.keys()]
            ax.pie(list(filtered.values()), labels=wedge_labels, colors=wedge_colors,
                   autopct='%1.0f%%', startangle=90,
                   textprops={'fontsize': 8})
            ax.set_title(f'{condition} Distraction\nRecovery Shapes', fontweight='bold')

        fig.suptitle('Recovery Shape Distribution\n(V = healthy rebound, L = attentional overwhelm)',
                     fontweight='bold', fontsize=12)
        plt.tight_layout()
        return self._save_or_show(fig, save_dir, 'plot6_recovery_shapes.png')

    def plot_cumulative_debt(self, debt_by_session, save_dir=None):
        """
        Plot 7: Cumulative attentional debt across trial sequence.
        Compares debt accumulation between baseline, active, and passive distraction.
        """
        if not debt_by_session:
            return

        condition_map = {1: ('Baseline', '#4C9BE8'),
                         2: ('Active Distraction', '#E84C4C'),
                         3: ('Passive Distraction', '#E8A94C')}

        fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
        fig.suptitle('Cumulative Attentional Debt Across Session\n'
                     '(Rising curve = unrecovered attentional cost accumulating)',
                     fontsize=13, fontweight='bold')

        for ax_idx, level_id in enumerate([1, 2, 3]):
            ax = axes[ax_idx]
            for set_id in [1, 2, 3]:
                key = (level_id, set_id)
                if key not in debt_by_session:
                    continue
                debts = debt_by_session[key]
                if not debts:
                    continue
                min_len = min(len(d) for d in debts)
                arr = np.array([d[:min_len] for d in debts])
                mean_debt = np.nanmean(arr, axis=0)
                label, color = condition_map[set_id]
                ax.plot(np.arange(len(mean_debt)), mean_debt,
                        color=color, label=label, linewidth=2)
                ax.fill_between(np.arange(len(mean_debt)),
                                mean_debt - np.nanstd(arr, axis=0),
                                mean_debt + np.nanstd(arr, axis=0),
                                alpha=0.1, color=color)

            ax.set_title(f'Level {level_id}', fontweight='bold')
            ax.set_xlabel('Trial Number')
            ax.set_ylabel('Cumulative Debt (s)') if ax_idx == 0 else None
            ax.grid(alpha=0.3, linestyle='--')
            ax.legend(fontsize=8)

        plt.tight_layout()
        return self._save_or_show(fig, save_dir, 'plot7_cumulative_debt.png')

    def plot_expanded_behavioral_metrics(self, expanded_metrics_df, save_dir=None):
        """
        Plot 8: Radar/spider chart of expanded behavioral metrics per condition.
        Shows IES, PES, error burstiness, micro-lapse rate, RT slope.
        """
        if expanded_metrics_df is None or len(expanded_metrics_df) == 0:
            return

        metrics_to_plot = ['IES', 'post_error_slowing', 'error_burstiness',
                           'micro_lapse_rate', 'rt_slope']
        metric_labels = ['Inv. Efficiency\n(IES)', 'Post-Error\nSlowing',
                         'Error\nBurstiness', 'Micro-Lapse\nRate', 'RT\nSlope']

        conditions = ['Baseline', 'Distraction+Response', 'Distraction+NoResponse']
        colors = {'Baseline': '#4C9BE8', 'Distraction+Response': '#E84C4C',
                  'Distraction+NoResponse': '#E8A94C'}

        # Normalize each metric 0–1 across all conditions for radar
        norm_df = expanded_metrics_df.copy()
        for m in metrics_to_plot:
            if m in norm_df.columns:
                col = norm_df[m].abs()  # abs for direction-agnostic normalization
                mn, mx = col.min(), col.max()
                if mx > mn:
                    norm_df[m] = (col - mn) / (mx - mn)
                else:
                    norm_df[m] = 0.5

        n_metrics = len(metrics_to_plot)
        angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
        angles += angles[:1]  # close the polygon

        fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                                 subplot_kw=dict(polar=True))
        fig.suptitle('Expanded Behavioral Metrics per Level (Normalized)\n'
                     'Larger area = more disrupted attention profile',
                     fontsize=12, fontweight='bold')

        for ax_idx, level_id in enumerate([1, 2, 3]):
            ax = axes[ax_idx]
            level_norm = norm_df[norm_df['Level'] == level_id] if 'Level' in norm_df.columns else norm_df

            for condition in conditions:
                cond_data = level_norm[level_norm['Condition'] == condition] if 'Condition' in level_norm.columns else level_norm
                if len(cond_data) == 0:
                    continue
                values = [cond_data[m].mean() if m in cond_data.columns else 0
                          for m in metrics_to_plot]
                values += values[:1]
                ax.plot(angles, values, color=colors[condition],
                        linewidth=2, label=condition)
                ax.fill(angles, values, color=colors[condition], alpha=0.15)

            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(metric_labels, fontsize=8)
            ax.set_title(f'Level {level_id}', fontweight='bold', pad=15)
            ax.set_ylim(0, 1)
            ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=7)

        plt.tight_layout()
        return self._save_or_show(fig, save_dir, 'plot8_expanded_metrics_radar.png')

    def plot_distraction_cost_trajectory(self, cost_trajectory_by_session, save_dir=None):
        """
        Plot 10: Distraction cost of each successive event across the session.
        Shows whether each distraction is MORE damaging than the last (fatigue)
        or LESS (habituation).
        """
        if not cost_trajectory_by_session:
            return

        fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
        fig.suptitle('Distraction Cost Trajectory Across Session\n'
                     '(Rising = each distraction more damaging than last → fatigue)',
                     fontsize=12, fontweight='bold')

        colors = {2: '#E84C4C', 3: '#E8A94C'}
        labels = {2: 'Active Distraction', 3: 'Passive Distraction'}

        for ax_idx, level_id in enumerate([1, 2, 3]):
            ax = axes[ax_idx]
            for set_id in [2, 3]:
                key = (level_id, set_id)
                if key not in cost_trajectory_by_session:
                    continue
                costs = cost_trajectory_by_session[key]
                if not costs:
                    continue
                min_len = min(len(c) for c in costs)
                arr = np.array([c[:min_len] for c in costs])
                mean_cost = np.nanmean(arr, axis=0)
                x = np.arange(1, len(mean_cost) + 1)
                ax.plot(x, mean_cost, color=colors[set_id],
                        marker='o', linewidth=2, markersize=5,
                        label=labels[set_id])
                ax.fill_between(x, mean_cost - np.nanstd(arr, axis=0),
                                mean_cost + np.nanstd(arr, axis=0),
                                alpha=0.12, color=colors[set_id])

                # Trend line
                if len(mean_cost) >= 3:
                    s, b, _, p, _ = linregress(x, mean_cost)
                    ax.plot(x, s * x + b, '--', color=colors[set_id], linewidth=1, alpha=0.6)
                    trend = '↑ fatigue' if s > 0 else '↓ habituation'
                    ax.text(x[-1], mean_cost[-1] + 0.01, trend,
                            fontsize=7, color=colors[set_id], ha='right')

            ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
            ax.set_title(f'Level {level_id}', fontweight='bold')
            ax.set_xlabel('Distraction Event #')
            ax.set_ylabel('RT Cost (s)') if ax_idx == 0 else None
            ax.grid(alpha=0.3, linestyle='--')
            ax.legend(fontsize=8)

        plt.tight_layout()
        return self._save_or_show(fig, save_dir, 'plot10_distraction_cost_trajectory.png')

    def _save_or_show(self, fig, save_dir, filename):
        """Save figure to file. Returns True if actually written to disk, else False."""
        saved = False
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            path = os.path.join(save_dir, filename)
            try:
                plt.figure(fig.number)
                plt.savefig(path, dpi=150, bbox_inches='tight')
                if os.path.exists(path) and os.path.getsize(path) > 0:
                    print(f"   Saved: {path}")
                    saved = True
                else:
                    print(f"   Warning: {filename} — file empty or missing after save")
            except Exception as e:
                import traceback
                print(f"   Warning: could not save {filename}: {e}")
                traceback.print_exc()
        plt.close(fig)
        return saved

    # ══════════════════════════════════════════════════════════════════════════
    # SUBJECT ANALYSIS
    # ══════════════════════════════════════════════════════════════════════════

    def analyze_subject(self, subject_id, level_id=1, task_id=1, set_id=1,
                        baseline_metrics=None, baseline_behavioral_data=None):
        """
        Complete behavioral analysis for one subject / level / task / set.
        EEG completely absent. Returns results dict.
        """
        task_name = self.get_task_name(level_id, task_id)
        display_name = self.get_display_name(task_name)
        is_baseline = (set_id == 1)

        print(f"\n{'=' * 60}")
        print(f"ANALYZING: Subject {subject_id} | Level {level_id} | "
              f"Task {task_id} ({display_name}) | Set {set_id}")
        print('=' * 60)

        behavioral_data = self.load_behavioral_data(subject_id, task_name, set_id, level_id)
        if behavioral_data is None:
            return None

        # Expanded behavioral metrics
        ext_metrics = self.compute_all_behavioral_metrics(
            behavioral_data,
            baseline_data=baseline_behavioral_data if not is_baseline else None
        )

        # Recovery
        recovery_time = np.nan
        peri_distraction = {}
        debt_series = np.array([])
        cost_trajectory = []

        if not is_baseline:
            recovery_time = self.compute_behavioral_recovery(behavioral_data)
            peri_distraction = self.compute_peri_distraction_metrics(behavioral_data)

            # Cumulative debt
            b_rt = (baseline_behavioral_data['rt'].mean()
                    if baseline_behavioral_data is not None
                       and 'rt' in baseline_behavioral_data.columns
                    else None)
            debt_series, peak_debt = self.compute_cumulative_debt(behavioral_data, baseline_rt=b_rt)
            ext_metrics['peak_cumulative_debt'] = peak_debt

            # Per-event distraction cost
            rt_col = 'rt' if 'rt' in behavioral_data.columns else 'reaction_time'
            if rt_col in behavioral_data.columns and 'stim_time' in behavioral_data.columns:
                for ev in peri_distraction.get('per_event', []):
                    pre_rt = ev.get('pre_rt_mean', np.nan)
                    first_rt = ev.get('first_post_rt', np.nan)
                    if not np.isnan(pre_rt) and not np.isnan(first_rt) and pre_rt > 0:
                        cost_trajectory.append(float(first_rt - pre_rt))

        # Trials-to-3-consecutive-correct (mean across distraction events)
        t3cc_result = self.compute_trials_to_consecutive_correct(behavioral_data)
        t3cc_mean = t3cc_result.get('mean', np.nan) if not is_baseline else np.nan

        # BAPS_s, BAPS_3CC, BAPS_Extended
        baps_result = self.compute_baps(
            behavioral_data,
            recovery_time=recovery_time,
            trials_to_3cc=t3cc_mean,
            baseline_metrics=baseline_metrics,
            is_baseline=is_baseline,
            extended_metrics=ext_metrics,
        )

        if is_baseline:
            baps_s, baps_3cc, baps_ext, baseline_metrics_out = baps_result
            # Cache IES in baseline metrics so Extended score can normalize later
            baseline_metrics_out['IES'] = ext_metrics.get('IES', np.nan)
        else:
            baps_s, baps_3cc, baps_ext = baps_result
            baseline_metrics_out = None

        # Print summary
        print(f"\nBEHAVIORAL SUMMARY:")
        print(f"  Accuracy:         {ext_metrics.get('accuracy', np.nan):.1%}")
        print(f"  Mean RT:          {ext_metrics.get('rt_mean', np.nan):.3f}s")
        print(f"  RT Variability:   {ext_metrics.get('rt_std', np.nan):.3f}s")
        print(f"  IES:              {ext_metrics.get('IES', np.nan):.3f}")
        print(f"  Post-Error Slow:  {ext_metrics.get('post_error_slowing', np.nan):.3f}s")
        print(f"  Micro-Lapse Rate: {ext_metrics.get('micro_lapse_rate', np.nan):.1%}")
        print(f"  RT Slope:         {ext_metrics.get('rt_slope', np.nan):.5f} s/trial")
        print(f"  Error Burstiness: {ext_metrics.get('error_burstiness', np.nan):.3f}")
        if not is_baseline:
            print(f"  Recovery Time:    {recovery_time:.3f}s")
            print(f"  Peak Debt:        {ext_metrics.get('peak_cumulative_debt', np.nan):.3f}s")
            peri_n = peri_distraction.get('n_events', 0)
            peri_rate = peri_distraction.get('recovery_completeness_rate', np.nan)
            print(f"  Recovery Complete Rate: {peri_rate:.1%} ({peri_n} events)")
        print(f"\n  BAPS_s        = {baps_s:.2f}%   (Accuracy + RT + RT_SD + Recovery_s)")
        print(f"  BAPS_3CC      = {baps_3cc:.2f}%   (Accuracy + RT + RT_SD + Recovery_3CC)")
        print(f"  BAPS_Extended = {baps_ext:.2f}%   (BAPS_s + IES + PostError + Lapses + Burstiness)")

        return {
            'behavioral_data': behavioral_data,
            'extended_metrics': ext_metrics,
            'recovery_time': recovery_time,
            'trials_to_3cc': t3cc_mean,
            'peri_distraction': peri_distraction,
            'debt_series': debt_series,
            'cost_trajectory': cost_trajectory,
            'BAPS': baps_s,
            'BAPS_3CC': baps_3cc,
            'BAPS_Extended': baps_ext,
            'baseline_metrics': baseline_metrics_out,
            'is_baseline': is_baseline,
        }


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════


In [ ]:
def _parse_time_to_seconds(time_str):
    """Convert '1:30pm' or '90' to seconds from midnight (float)."""
    s = time_str.strip().lower().replace(' ', '')
    if 'pm' in s or 'am' in s:
        for fmt in ('%I:%M%p', '%I%p'):
            try:
                t = datetime.strptime(s, fmt).time()
                return t.hour * 3600 + t.minute * 60 + t.second
            except ValueError:
                continue
        return None
    try:
        return float(s)
    except ValueError:
        return None


In [ ]:
def _spread_duplicate_times(time_list):
    """
    When multiple stimuli fall in the same clock-minute, spread them evenly
    across that 60-second window so no two entries map to the same second.

    Examples
    --------
    2 at 10:53am → [10:53:00, 10:53:30]   (30 s apart)
    3 at 10:53am → [10:53:00, 10:53:20, 10:53:40]  (20 s apart)
    N at same minute → every 60/N seconds within that minute
    """
    if not time_list:
        return time_list

    from collections import defaultdict

    # Group original positions by minute bucket
    buckets = defaultdict(list)
    for i, t in enumerate(time_list):
        buckets[int(t // 60)].append(i)

    spread = list(time_list)
    for minute, indices in buckets.items():
        n = len(indices)
        if n > 1:
            base = minute * 60.0
            interval = 60.0 / n
            # Sort indices by their original time so spreading is ordered
            indices_sorted = sorted(indices, key=lambda i: time_list[i])
            for rank, idx in enumerate(indices_sorted):
                spread[idx] = base + rank * interval

    return spread


In [ ]:
def _convert_times_to_relative(time_list):
    """Return times relative to the first (smallest) time in the list."""
    if not time_list:
        return []
    base = min(time_list)
    return [t - base for t in sorted(time_list)]


In [ ]:
def main():
    print("=" * 70)
    print("BEHAVIORAL ATTENTION ANALYSIS — NO EEG")
    print("Behavioral Attention Performance Score (BAPS)")
    print("=" * 70)
    print("\nThis analysis includes:")
    print("  LEVEL 1: Classic, SART, Semantic")
    print("  LEVEL 2: Selective Visual, Color-Number, Spatial Cue")
    print("  LEVEL 3: Picture Recall, Color-Location, Dual Task")
    print("\nMetrics computed:")
    print("  Core:     Accuracy, RT, RT_SD, Recovery Time, IES, RCS")
    print("  Extended: Post-Error Slowing, Error Burstiness, Micro-Lapse Rate,")
    print("            RT Slope, Cumulative Debt, Distraction Cost Trajectory,")
    print("            Recovery Shape (V/U/L), Anticipatory Drift")
    print("=" * 70)

    analyzer = BehavioralAttentionAnalyzer()

    # ── STEP 1: Level Selection ───────────────────────────────────────────────
    print("\nSTEP 1: Level Selection")
    print("-" * 30)
    print("1. Level 1 only")
    print("2. Level 2 only")
    print("3. Level 3 only")
    print("4. All Levels")
    while True:
        lc = input("Enter choice (1-4): ").strip()
        if lc in ['1', '2', '3', '4']:
            selected_levels = [int(lc)] if lc != '4' else [1, 2, 3]
            break
        print("Please enter 1, 2, 3, or 4.")

    # ── STEP 2: Subject Selection ─────────────────────────────────────────────
    print("\nSTEP 2: Subject Selection")
    print("-" * 30)
    subject_ids_per_level = {}
    for level_id in selected_levels:
        avail = analyzer.find_available_subjects(level_id)
        csv_path = analyzer.get_csv_path_for_level(level_id)
        if avail:
            print(f"\n  Available subjects for Level {level_id}: {avail}")
        else:
            print(f"\n  CSV path for Level {level_id}: {csv_path}")
        while True:
            try:
                sid = int(input(f"  Enter Subject ID for Level {level_id} (e.g., 5): ").strip())
                subject_ids_per_level[level_id] = sid
                csv_dir = analyzer.get_csv_path_for_level(level_id, sid)
                if os.path.exists(csv_dir):
                    print(f"  Found data at: {csv_dir}")
                else:
                    print(f"  Warning: {csv_dir} not found — will search alternative paths.")
                break
            except ValueError:
                print("  Please enter a valid number.")

    # ── STEP 3: Stimulus Times Configuration ─────────────────────────────────
    print("\nSTEP 3: Stimulus Times Configuration")
    print("-" * 40)
    print("Enter stimulus times for Sets 2 & 3 (Set 1 is Baseline — skipped).")
    print("Supported formats:")
    print("  • Clock time: 1:30pm, 1:31pm, 1:32pm")
    print("  • Seconds:    0, 60, 120, 180")
    print("Press Enter to skip (no distraction stimuli recorded).")

    stim_times_store = {}   # (level_id, task_id, set_id) → {'absolute': [...], 'relative': [...]}

    for level_id in selected_levels:
        task_names = analyzer.task_names_by_level[level_id]

        print(f"\n{'='*50}")
        print(f"LEVEL {level_id} STIMULUS TIMES")
        print(f"{'='*50}")

        for set_id in [1, 2, 3]:
            set_label = {
                1: "SET 1: BASELINE (No Distraction)",
                2: "SET 2: DISTRACTION + RESPONSE",
                3: "SET 3: DISTRACTION + NO RESPONSE",
            }[set_id]
            print(f"\n{set_label}")
            print("-" * 30)

            for task_id in [1, 2, 3]:
                tn = task_names[task_id]
                dn = analyzer.get_display_name(tn)

                if set_id == 1:
                    print(f"  {dn} (Task {task_id}): Baseline — no stimulus times needed")
                    stim_times_store[(level_id, task_id, set_id)] = {'absolute': [], 'relative': []}
                    continue

                while True:
                    raw = input(f"  {dn} (Task {task_id}) Set {set_id}: ").strip()
                    if not raw:
                        stim_times_store[(level_id, task_id, set_id)] = {'absolute': [], 'relative': []}
                        print(f"    No stimulus times recorded.")
                        break
                    tokens = [t.strip() for t in raw.split(',')]
                    parsed = [_parse_time_to_seconds(t) for t in tokens]
                    if any(v is None for v in parsed):
                        bad = [t for t, v in zip(tokens, parsed) if v is None]
                        print(f"    Could not parse: {bad}. Try '1:30pm,1:31pm' or '0,60,120'.")
                        continue
                    # Spread duplicates that land in the same clock-minute
                    spread = _spread_duplicate_times(parsed)
                    relative = _convert_times_to_relative(spread)
                    stim_times_store[(level_id, task_id, set_id)] = {
                        'absolute': spread,
                        'relative': relative,
                    }
                    print(f"    {len(spread)} stimulus times:")
                    print(f"      Input:    {tokens}")
                    # Show spread note if any minute had duplicates
                    if spread != parsed:
                        print(f"      Raw abs:  {[f'{t:.0f}s' for t in parsed]}")
                        print(f"      Spread:   {[f'{t:.0f}s' for t in spread]}  ← duplicates spread within minute")
                    else:
                        print(f"      Absolute: {[f'{t:.0f}s' for t in spread]}")
                    print(f"      Relative: {[f'{t:.1f}s' for t in relative]}")
                    break

    _run_analysis_pipeline(analyzer, selected_levels, subject_ids_per_level, stim_times_store)


In [ ]:
def _run_analysis_pipeline(analyzer, selected_levels, subject_ids_per_level, stim_times_store):
    """
    Execute Steps 4–8: inject stim times into CSVs, run analysis, save results.
    Called by main() after interactive input collection, and by run_batch_analysis.py
    for fully automated batch processing.
    """
    # ── STEP 4: Inject stim_time column into CSVs ────────────────────────────
    print("\nSTEP 4: Syncing Stimulus Times with Behavioral CSVs")
    print("-" * 50)

    for level_id in selected_levels:
        subject_id = subject_ids_per_level[level_id]
        task_names = analyzer.task_names_by_level[level_id]
        csv_dir = analyzer.get_csv_path_for_level(level_id, subject_id)

        # Search multiple candidate paths (mirrors v3 approach)
        csv_paths_to_try = [
            csv_dir,
            os.path.join(analyzer.base_path, f"Level{level_id}/Results_CSV/Subject{subject_id}"),
            os.path.join(analyzer.base_path, f"Level{level_id}/Results_CSV/subject{subject_id}"),
        ]

        print(f"\nLevel {level_id} — Subject {subject_id}")

        for task_id in [1, 2, 3]:
            tn = task_names[task_id]
            dn = analyzer.get_display_name(tn)

            for set_id in [1, 2, 3]:
                entry = stim_times_store.get((level_id, task_id, set_id), {})
                stim_abs = entry.get('absolute', [])
                stim_rel = entry.get('relative', [])

                # Locate the CSV file
                patterns = [
                    f"{tn}_results_set{set_id}_",
                    f"{tn.lower()}_results_set{set_id}_",
                    f"{tn.capitalize()}_results_set{set_id}_",
                ]
                csv_fp = None
                for base in csv_paths_to_try:
                    if not os.path.exists(base):
                        continue
                    for p in patterns:
                        hits = [f for f in os.listdir(base) if p in f and f.endswith('.csv')]
                        if hits:
                            csv_fp = os.path.join(base, sorted(hits)[-1])
                            break
                    if csv_fp:
                        break

                if csv_fp is None:
                    continue

                try:
                    df = pd.read_csv(csv_fp)
                    df['stim_time'] = np.nan
                    stim_times_to_use = stim_abs if stim_abs else stim_rel

                    if not stim_times_to_use:
                        df.to_csv(csv_fp, index=False)
                        continue

                    matched = 0
                    if 'timestamp' in df.columns:
                        try:
                            csv_ts = pd.to_datetime(df['timestamp'], errors='coerce')
                            t0 = csv_ts.iloc[0]

                            # Use absolute (clock-time) matching when available
                            if stim_abs:
                                from_midnight = (
                                    csv_ts.dt.hour * 3600
                                    + csv_ts.dt.minute * 60
                                    + csv_ts.dt.second
                                    + csv_ts.dt.microsecond / 1e6
                                ).values
                                taken = set()
                                for abs_t, rel_t in zip(stim_abs, stim_rel):
                                    diffs = np.abs(from_midnight - abs_t)
                                    for idx in taken:
                                        diffs[idx] = np.inf
                                    best = int(np.argmin(diffs))
                                    if diffs[best] <= 300:
                                        df.loc[best, 'stim_time'] = rel_t
                                        taken.add(best)
                                        matched += 1
                            else:
                                # Relative matching from session start
                                offsets = (csv_ts - t0).dt.total_seconds().values
                                taken = set()
                                for rel_t in stim_rel:
                                    diffs = np.abs(offsets - rel_t)
                                    for idx in taken:
                                        diffs[idx] = np.inf
                                    best = int(np.argmin(diffs))
                                    if diffs[best] <= 300:
                                        df.loc[best, 'stim_time'] = rel_t
                                        taken.add(best)
                                        matched += 1
                        except Exception as e:
                            print(f"    Timestamp sync error ({dn} Set {set_id}): {e} — index fallback")
                            idxs = np.linspace(0, len(df) - 1, len(stim_times_to_use), dtype=int)
                            for i, st in enumerate(stim_times_to_use):
                                df.loc[int(idxs[i]), 'stim_time'] = st
                            matched = len(stim_times_to_use)
                    else:
                        idxs = np.linspace(0, len(df) - 1, len(stim_times_to_use), dtype=int)
                        for i, st in enumerate(stim_times_to_use):
                            df.loc[int(idxs[i]), 'stim_time'] = st
                        matched = len(stim_times_to_use)

                    df.to_csv(csv_fp, index=False)
                    print(f"  {dn} Set {set_id}: matched {matched}/{len(stim_times_to_use)} stimuli "
                          f"→ {os.path.basename(csv_fp)}")
                except Exception as e:
                    print(f"  {dn} Set {set_id}: save error — {e}")

    # ── STEP 5: Run Analysis ──────────────────────────────────────────────────
    print("\nSTEP 5: Running Analysis")
    print("-" * 40)

    all_results = {}
    results_table = []
    debt_by_session = {}
    cost_trajectory_by_session = {}
    peri_distraction_results = {}
    behavioral_data_by_session = {}

    baseline_metrics_store = {}
    baseline_behavioral_store = {}

    condition_map = {1: 'Baseline', 2: 'Distraction+Response', 3: 'Distraction+NoResponse'}

    for level_id in selected_levels:
        subject_id = subject_ids_per_level[level_id]
        task_names = analyzer.task_names_by_level[level_id]

        for set_id in [1, 2, 3]:
            for task_id in [1, 2, 3]:
                bkey = (level_id, task_id)
                b_metrics = baseline_metrics_store.get(bkey) if set_id > 1 else None
                b_beh = baseline_behavioral_store.get(bkey) if set_id > 1 else None

                results = analyzer.analyze_subject(
                    subject_id=subject_id, level_id=level_id,
                    task_id=task_id, set_id=set_id,
                    baseline_metrics=b_metrics,
                    baseline_behavioral_data=b_beh
                )

                if results is None:
                    continue

                tn = task_names[task_id]
                dn = analyzer.get_display_name(tn)
                em = results['extended_metrics']
                condition = condition_map[set_id]
                session_key = (level_id, set_id, tn)

                all_results[session_key] = results

                # Cache baseline
                if set_id == 1 and results['baseline_metrics']:
                    baseline_metrics_store[bkey] = results['baseline_metrics']
                    baseline_behavioral_store[bkey] = results['behavioral_data']

                # Collect for plots
                ls_key = (level_id, set_id)
                if ls_key not in behavioral_data_by_session:
                    behavioral_data_by_session[ls_key] = []
                if results['behavioral_data'] is not None:
                    behavioral_data_by_session[ls_key].append(results['behavioral_data'])

                if len(results['debt_series']) > 0:
                    if ls_key not in debt_by_session:
                        debt_by_session[ls_key] = []
                    debt_by_session[ls_key].append(results['debt_series'])

                if results['cost_trajectory']:
                    if ls_key not in cost_trajectory_by_session:
                        cost_trajectory_by_session[ls_key] = []
                    cost_trajectory_by_session[ls_key].append(results['cost_trajectory'])

                if results['peri_distraction']:
                    peri_distraction_results[session_key] = results['peri_distraction']

                results_table.append({
                    'Level': level_id,
                    'Set': set_id,
                    'Condition': condition,
                    'Task': dn,
                    'Participant': subject_id,
                    'BAPS_s': results['BAPS'],
                    'BAPS_3CC': results.get('BAPS_3CC', np.nan),
                    'BAPS_Extended': results.get('BAPS_Extended', np.nan),
                    'Trials_to_3CC': results.get('trials_to_3cc', np.nan),
                    'Accuracy_%': em.get('accuracy', np.nan) * 100,
                    'RT_Mean_s': em.get('rt_mean', np.nan),
                    'RT_Std_s': em.get('rt_std', np.nan),
                    'IES': em.get('IES', np.nan),
                    'RCS': em.get('RCS', np.nan),
                    'Post_Error_Slowing': em.get('post_error_slowing', np.nan),
                    'Error_Burstiness': em.get('error_burstiness', np.nan),
                    'Micro_Lapse_Rate': em.get('micro_lapse_rate', np.nan),
                    'RT_Slope': em.get('rt_slope', np.nan),
                    'Recovery_s': results['recovery_time'],
                    'Peak_Debt_s': em.get('peak_cumulative_debt', np.nan),
                    'Distraction_Cost_RT%': em.get('distraction_cost_rt', np.nan),
                })

    # ── STEP 6: Print results table ───────────────────────────────────────────
    print("\nSTEP 6: Results Summary")
    print("=" * 140)
    results_df = pd.DataFrame(results_table)

    display_cols = ['Level', 'Set', 'Condition', 'Task',
                    'BAPS_s', 'BAPS_3CC', 'BAPS_Extended',
                    'Accuracy_%', 'RT_Mean_s', 'RT_Std_s', 'IES',
                    'Post_Error_Slowing', 'Error_Burstiness',
                    'Micro_Lapse_Rate', 'RT_Slope',
                    'Recovery_s', 'Trials_to_3CC', 'Peak_Debt_s']
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.float_format', '{:.3f}'.format)
    print(results_df[[c for c in display_cols if c in results_df.columns]].to_string(index=False))

    # ── STEP 7: Generate all plots ────────────────────────────────────────────
    primary_subject_id = list(subject_ids_per_level.values())[0]
    subject_info = ", ".join([f"L{l}:S{s}" for l, s in subject_ids_per_level.items()])

    print(f"\nSTEP 7: Generating Plots")
    print("-" * 40)
    save_dir = os.path.join(analyzer.base_path, f"plots_Subject{primary_subject_id}")
    os.makedirs(save_dir, exist_ok=True)
    print(f"Saving all plots to: {save_dir}/")

    plot_calls = [
        ("plot1_baps_by_condition",        lambda: analyzer.plot_baps_by_condition(results_df, save_dir=save_dir)),
        ("plot2_heatmap",                  lambda: analyzer.plot_individual_heatmap(results_df, save_dir=save_dir)),
        ("plot3_recovery_time",            lambda: analyzer.plot_recovery_time(results_df, save_dir=save_dir)),
        ("plot4_performance_trajectory",   lambda: analyzer.plot_performance_trajectory(behavioral_data_by_session, save_dir=save_dir)),
        ("plot5_recovery_shape",           lambda: analyzer.plot_recovery_shape_distribution(peri_distraction_results, save_dir=save_dir)),
        ("plot6_cumulative_debt",          lambda: analyzer.plot_cumulative_debt(debt_by_session, save_dir=save_dir)),
        ("plot7_expanded_metrics",         lambda: analyzer.plot_expanded_behavioral_metrics(results_df, save_dir=save_dir)),
        ("plot8_distraction_cost",         lambda: analyzer.plot_distraction_cost_trajectory(cost_trajectory_by_session, save_dir=save_dir)),
    ]
    plots_saved = 0
    for plot_name, plot_fn in plot_calls:
        try:
            result = plot_fn()
            if result is True:
                plots_saved += 1
        except Exception as e:
            import traceback
            print(f"   Warning: {plot_name} failed — {e}")
            traceback.print_exc()
    actual = len([f for f in os.listdir(save_dir) if f.endswith('.png')])
    print(f"   {actual} PNG files written to {save_dir}/")

    # ── STEP 8: Save results ──────────────────────────────────────────────────
    # CSV (main results — used by standalone plot script)
    out_csv = os.path.join(analyzer.base_path, f"BAPS_Subject{primary_subject_id}_results.csv")
    results_df.to_csv(out_csv, index=False)

    # Save additional data needed by standalone plot script
    _sid = primary_subject_id
    # Trial-level RT (for performance trajectory plot)
    trial_rows = []
    for (lvl, st), dfs in behavioral_data_by_session.items():
        for df in dfs:
            rt_col = 'rt' if 'rt' in df.columns else ('reaction_time' if 'reaction_time' in df.columns else None)
            acc_col = analyzer.get_accuracy_column(df) if df is not None and len(df) > 0 else None
            if rt_col and len(df) > 0:
                stim_col = next(
                    (c for c in df.columns if str(c).lower() == 'stim_time'),
                    None,
                )
                for i, row in df.reset_index(drop=True).iterrows():
                    at_dist = 0
                    if stim_col is not None:
                        v = row.get(stim_col, np.nan)
                        if pd.notna(v) and str(v).strip().lower() not in ('', 'nan', 'none'):
                            at_dist = 1
                    trial_rows.append(
                        {
                            'Level': lvl,
                            'Set': st,
                            'Trial': i,
                            'RT': row.get(rt_col, float('nan')),
                            'Correct': (
                                row.get(acc_col, float('nan')) if acc_col else float('nan')
                            ),
                            'At_distraction': at_dist,
                        }
                    )
    if trial_rows:
        pd.DataFrame(trial_rows).to_csv(
            os.path.join(analyzer.base_path, f"BAPS_Subject{_sid}_trial_rt.csv"), index=False)

    # Cumulative debt series (for debt plot)
    debt_rows = []
    for (lvl, st), debts in debt_by_session.items():
        for d_idx, debt in enumerate(debts):
            for t_idx, val in enumerate(debt):
                debt_rows.append({'Level': lvl, 'Set': st, 'Series': d_idx, 'Trial': t_idx, 'Debt_s': val})
    if debt_rows:
        pd.DataFrame(debt_rows).to_csv(
            os.path.join(analyzer.base_path, f"BAPS_Subject{_sid}_debt.csv"), index=False)

    # Distraction cost trajectory (for cost plot)
    cost_rows = []
    for (lvl, st), costs in cost_trajectory_by_session.items():
        for d_idx, cost in enumerate(costs):
            for e_idx, val in enumerate(cost):
                cost_rows.append({'Level': lvl, 'Set': st, 'Series': d_idx, 'Event': e_idx, 'Cost_s': val})
    if cost_rows:
        pd.DataFrame(cost_rows).to_csv(
            os.path.join(analyzer.base_path, f"BAPS_Subject{_sid}_cost.csv"), index=False)

    # Recovery shapes (for pie chart plot)
    shape_rows = []
    for (lvl, st, task), peri in peri_distraction_results.items():
        dist = peri.get('shape_distribution', {})
        for shape, count in dist.items():
            shape_rows.append({'Level': lvl, 'Set': st, 'Task': task, 'Shape': shape, 'Count': count})
    if shape_rows:
        pd.DataFrame(shape_rows).to_csv(
            os.path.join(analyzer.base_path, f"BAPS_Subject{_sid}_shapes.csv"), index=False)

    print(f"\n   Plot data CSVs saved — run plot_baps_results.py to generate plots:")
    print(f"     python3 plot_baps_results.py {out_csv}")

    # Text summary (mirrors v3 Analysis_Subject{N}_trialMappedRecovery.txt style)
    out_txt = os.path.join(analyzer.base_path, f"BAPS_Subject{primary_subject_id}_Analysis.txt")
    try:
        with open(out_txt, 'w') as f:
            f.write("=" * 120 + "\n")
            f.write(f"BEHAVIORAL ATTENTION ANALYSIS (BAPS) — SUBJECT {primary_subject_id}\n")
            f.write(f"Subjects analyzed: {subject_info}\n")
            f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("=" * 120 + "\n\n")

            f.write("SCORE DEFINITIONS:\n")
            f.write("-" * 60 + "\n")
            f.write("  BAPS_s        = Accuracy + RT + RT_SD + Recovery_s  (4 components)\n")
            f.write("                  Recovery_s = elapsed seconds until rolling RT window\n")
            f.write("                  returns to within 10% of pre-distraction RT mean.\n")
            f.write("                  Scale: 0 s → score 1.0,  ≥5 s → score 0.0  (linear decay).\n\n")
            f.write("  BAPS_3CC      = Accuracy + RT + RT_SD + Recovery_3CC  (4 components)\n")
            f.write("                  Recovery_3CC = mean trials to 3 consecutive correct\n")
            f.write("                  responses after each distraction event in the set.\n")
            f.write("                  Normalization: score = 3 / max(mean_trials, 3)\n")
            f.write("                    3 trials (immediate) → 1.00\n")
            f.write("                    6 trials             → 0.50\n")
            f.write("                    9 trials             → 0.33\n")
            f.write("                    15 trials            → 0.20\n")
            f.write("                    never recovered      → 0.00  (penalized)\n\n")
            f.write("  BAPS_Extended = BAPS_s base + IES + PostError + Lapses + Burstiness\n")
            f.write("                  Adds cognitive-efficiency and error-pattern metrics.\n\n")
            f.write("  All 3 shared sub-scores (Accuracy, RT, RT_SD) normalized against\n")
            f.write("  Set 1 (Baseline) values. Baseline = 100 for all three scores.\n")
            f.write("  Scores >100 = outperforms own baseline.\n\n")

            f.write("DETAILED RESULTS TABLE:\n")
            f.write("-" * 140 + "\n")
            table_cols = [c for c in display_cols if c in results_df.columns]
            f.write(results_df[table_cols].to_string(index=False) + "\n")
            f.write("-" * 140 + "\n\n")

            f.write("SUMMARY BY LEVEL AND SET:\n")
            f.write("-" * 60 + "\n")
            for level_id in selected_levels:
                level_df = results_df[results_df['Level'] == level_id]
                if len(level_df) == 0:
                    continue
                f.write(f"\nLevel {level_id}:\n")
                for set_id in [1, 2, 3]:
                    set_df = level_df[level_df['Set'] == set_id]
                    if len(set_df) == 0:
                        continue
                    bs   = set_df['BAPS_s'].mean()
                    b3cc = set_df['BAPS_3CC'].mean()
                    ext  = set_df['BAPS_Extended'].mean()
                    acc  = set_df['Accuracy_%'].mean()
                    t3cc_col = set_df['Trials_to_3CC'].dropna()
                    t3cc_str = f"{t3cc_col.mean():.1f}" if len(t3cc_col) else "N/A"
                    f.write(f"  Set {set_id}: BAPS_s = {bs:.1f},  BAPS_3CC = {b3cc:.1f},  "
                            f"BAPS_Extended = {ext:.1f},  "
                            f"Accuracy = {acc:.1f}%,  Trials_to_3CC (mean) = {t3cc_str}\n")

            # ── Interpretation guide ──────────────────────────────────────────
            f.write("\n\n" + "=" * 120 + "\n")
            f.write("METRIC INTERPRETATION GUIDE\n")
            f.write("=" * 120 + "\n\n")
            f.write("How to read each metric and judge participant performance:\n\n")

            guide = [
                ("BAPS_s (Accuracy + RT + RT_SD + Recovery_s)",
                 "100 = on-par with own baseline. >100 = improved. <100 = distraction cost.\n"
                 "       60–80 = moderate impairment. <60 = substantial distraction cost.\n"
                 "       Recovery component: time-based (seconds until RT returns to ≤10% above\n"
                 "       baseline). 0 s → contributes 1.0; ≥5 s → contributes 0.0.\n"
                 "       Use this to compare directly with EEG-based ARS scores."),

                ("BAPS_3CC (Accuracy + RT + RT_SD + Recovery_3CC)",
                 "Same accuracy/RT interpretation as BAPS_s but recovery is measured by\n"
                 "       how many trials until 3 consecutive correct responses after each distraction.\n"
                 "       Normalization: score = 3 / max(mean_trials, 3).\n"
                 "       BAPS_3CC > BAPS_s → participant is fast-to-correct (trials) but slow in clock-time.\n"
                 "       BAPS_3CC < BAPS_s → participant recovers quickly in time but takes many trials.\n"
                 "       Prefer BAPS_3CC when trial-count (error patterns) matter more than raw time."),

                ("BAPS_Extended",
                 "Same as BAPS_s base but penalised further by cognitive inefficiency (IES),\n"
                 "       post-error behaviour, lapse frequency, and error clustering.\n"
                 "       A gap between BAPS_s and Extended (BAPS_s >> Extended) means the participant\n"
                 "       maintained accuracy/speed but showed poor error-recovery or bursty errors."),

                ("Accuracy_%",
                 "% trials with the correct key press. Higher = better.\n"
                 "       ≥95%: excellent sustained attention.\n"
                 "       85–94%: mild lapses.\n"
                 "       <85%: significant attentional failure under distraction."),

                ("RT_Mean_s",
                 "Mean reaction time in seconds. Lower = faster responses.\n"
                 "       Increases under distraction indicate slowed processing.\n"
                 "       Compare Set 2/3 vs Set 1: >20% increase = meaningful slowing."),

                ("RT_Std_s",
                 "RT variability / inconsistency. Lower = more stable responses.\n"
                 "       High RT_Std with normal RT_Mean = fluctuating attention (some\n"
                 "       trials very fast, others very slow — hallmark of mind-wandering)."),

                ("IES (Inverse Efficiency Score)",
                 "RT_Mean / Accuracy. Lower = more efficient.\n"
                 "       Penalises speed-accuracy trade-offs: a fast-but-inaccurate\n"
                 "       participant scores poorly; slow-but-accurate also scores poorly.\n"
                 "       Rising IES in Sets 2 & 3 indicates cognitive load from distraction."),

                ("Post_Error_Slowing (s)",
                 "Mean extra RT on the trial immediately after an error.\n"
                 "       Positive = participant slows down after mistakes (adaptive monitoring).\n"
                 "       Near-zero or negative = no self-monitoring (impulsive or inattentive).\n"
                 "       Typical healthy adults show +50–200 ms PES."),

                ("Error_Burstiness",
                 "Coefficient of variation of inter-error intervals (−1 to +1 approx).\n"
                 "       Near 0 = errors distributed evenly (random lapses).\n"
                 "       >0 = errors cluster in bursts → attention collapses in waves.\n"
                 "       <0 = errors spread out more than chance → unusually regular."),

                ("Micro_Lapse_Rate",
                 "Fraction of trials where RT > 2× the participant's own mean.\n"
                 "       0–5%: normal. 5–15%: occasional attention failures.\n"
                 "       >15%: frequent micro-lapses → vigilance decrement under distraction."),

                ("RT_Slope (s/trial)",
                 "Linear trend of RT across trials within a session.\n"
                 "       Positive slope = fatigue / vigilance decrement over time.\n"
                 "       Negative slope = warm-up / learning effect.\n"
                 "       |slope| > 0.005 s/trial is typically meaningful."),

                ("Recovery_s",
                 "Time (seconds) until RT window returns to within 10% of pre-distraction RT.\n"
                 "       <2s: excellent resilience. 2–5s: moderate. >5s: poor recovery.\n"
                 "       NaN / penalised at 0 = performance never fully recovered."),

                ("Trials_to_3CC",
                 "Mean number of trials until 3 consecutive correct responses after a distraction.\n"
                 "       3 trials = achieved immediately (best possible).\n"
                 "       6–10 trials: moderate recovery. >15: slow trial-level recovery.\n"
                 "       NaN = never achieved 3CC within the remaining trials (severe disruption).\n"
                 "       Used directly in BAPS_3CC as the recovery component."),

                ("Peak_Debt_s",
                 "Cumulative attentional debt: sum of RT excesses over the whole session.\n"
                 "       Low values = distraction had minimal cumulative impact.\n"
                 "       High values = distraction degraded the entire session's efficiency."),
            ]
            for metric, interp in guide:
                f.write(f"  {metric}:\n")
                f.write(f"       {interp}\n\n")

            f.write("=" * 120 + "\n")
            f.write("ANALYSIS COMPLETE\n")
            f.write("=" * 120 + "\n")
        print(f"\n*** Results saved to: {out_txt} ***")
    except Exception as e:
        print(f"\nWarning: Could not save results file: {e}")

    print(f"CSV  saved to: {out_csv}")
    print(f"Plots saved to: {save_dir}/")
    print(f"\nCOMPLETE ANALYSIS FINISHED!")
    print(f"Subjects analyzed: {subject_info}")
    print("=" * 60)


In [ ]:
# Run this cell to execute the script
main()
